In [4]:
import pandas as pd
import requests
import time
import shutil

#import yaml
from pathlib import Path

In [4]:
config_path = Path.cwd().parent / "config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

root = Path(config['project_root'])

FileNotFoundError: [Errno 2] No such file or directory: 'd:\\Nueva carpeta\\AI-Based-Smart-City-Air-Quality-Monitoring-and-Forecasting-System-for-Greater-Bilbao\\config.yaml'

In [28]:
# Get weather data
stations = {
    "ALGORTA_BBIZI2": (43.362056, -3.022782),
    "BARAKALDO": (43.298379, -2.987133),
    "BASAURI": (43.241131, -2.883761),
    "ERANDIO": (43.302653, -2.977240),
    "MAZARREDO": (43.267506, -2.935188),
    "MUSKIZ": (43.320713, -3.112716),
    "SANTURCE": (43.333012, -3.042560)
}

for name, (lat, lon) in stations.items():
    print(f"Processing: {name}...")
    
    url = (
        "https://archive-api.open-meteo.com/v1/era5"
        f"?latitude={lat}&longitude={lon}"
        "&start_date=2015-01-01&end_date=2026-05-06"
        "&daily=temperature_2m_mean,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant"
        "&timezone=Europe/Madrid"
    )
    
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        
        if "daily" in data and "time" in data["daily"]:
            df = pd.DataFrame(data["daily"])
            
            df.rename(columns={
                "temperature_2m_mean": "Temperature",
                "relative_humidity_2m_mean": "Humidity",
                "precipitation_sum": "Precipitation",
                "wind_speed_10m_max": "WindSpeed",
                "wind_direction_10m_dominant": "WindDirection"
            }, inplace=True)
            
            df["Date"] = pd.to_datetime(df["time"])
            df.drop(columns=["time"], inplace=True)
            
            df.to_csv(root / "data" / "raw" / f"{name}_weather.csv", index=False)
            print(f"✅ Success: {name} saved.")
        else:
            print(f"❌ Error: Could not extract data for {name}.")
            
    except Exception as e:
        print(f"⚠️ Error for {name}: {e}")
    
    time.sleep(1.5) 

print("\nFinished!")

Processing: ALGORTA_BBIZI2...
✅ Success: ALGORTA_BBIZI2 saved.
Processing: BARAKALDO...
✅ Success: BARAKALDO saved.
Processing: BASAURI...
✅ Success: BASAURI saved.
Processing: ERANDIO...
✅ Success: ERANDIO saved.
Processing: MAZARREDO...
✅ Success: MAZARREDO saved.
Processing: MUSKIZ...
❌ Error: Could not extract data for MUSKIZ.
Processing: SANTURCE...
❌ Error: Could not extract data for SANTURCE.

Finished!


In [30]:
# Get weather data
stations = {

    "MUSKIZ": (43.320713, -3.112716),
    "SANTURCE": (43.333012, -3.042560)

}

for name, (lat, lon) in stations.items():
    print(f"Processing: {name}...")
    
    url = (
        "https://archive-api.open-meteo.com/v1/era5"
        f"?latitude={lat}&longitude={lon}"
        "&start_date=2015-01-01&end_date=2026-05-06"
        "&daily=temperature_2m_mean,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant"
        "&timezone=Europe/Madrid"
    )
    
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        
        if "daily" in data and "time" in data["daily"]:
            df = pd.DataFrame(data["daily"])
            
            df.rename(columns={
                "temperature_2m_mean": "Temperature",
                "relative_humidity_2m_mean": "Humidity",
                "precipitation_sum": "Precipitation",
                "wind_speed_10m_max": "WindSpeed",
                "wind_direction_10m_dominant": "WindDirection"
            }, inplace=True)
            
            df["Date"] = pd.to_datetime(df["time"])
            df.drop(columns=["time"], inplace=True)
            
            df.to_csv(root / "data" / "raw" / f"{name}_weather.csv", index=False)
            print(f"✅ Success: {name} saved.")
        else:
            print(f"❌ Error: Could not extract data for {name}.")
            
    except Exception as e:
        print(f"⚠️ Error for {name}: {e}")
    
    time.sleep(1.5) 

print("\nFinished!")

Processing: MUSKIZ...
✅ Success: MUSKIZ saved.
Processing: SANTURCE...
✅ Success: SANTURCE saved.

Finished!


In [31]:
raw_data_path = root / "data" / "raw"

all_files = list(raw_data_path.glob("*_weather.csv"))

combined_df = pd.DataFrame()

for file_path in all_files:
    station_name = file_path.stem.replace("_weather", "")
    
    df = pd.read_csv(file_path)
    
    df["Station"] = station_name
    
    combined_df = pd.concat([combined_df, df], ignore_index=True)

combined_df = combined_df.sort_values(by=["Date", "Station"])
combined_df.head()

,Temperature,Humidity,Precipitation,WindSpeed,WindDirection,Date,Station
0,6.4,85,0.0,8.0,176,2015-01-01,ALGORTA_BBIZI2
4144,6.7,85,0.0,8.0,176,2015-01-01,BARAKALDO
8288,4.7,82,0.0,8.0,176,2015-01-01,BASAURI
12432,6.8,85,0.0,8.0,176,2015-01-01,ERANDIO
16576,6.8,83,0.0,8.0,176,2015-01-01,MAZARREDO


In [32]:
combined_df.info()

<class 'pandas.DataFrame'>
Index: 33152 entries, 0 to 33151
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Temperature    33152 non-null  float64
 1   Humidity       33152 non-null  int64  
 2   Precipitation  33152 non-null  float64
 3   WindSpeed      33152 non-null  float64
 4   WindDirection  33152 non-null  int64  
 5   Date           33152 non-null  str    
 6   Station        33152 non-null  str    
dtypes: float64(3), int64(2), str(2)
memory usage: 2.6 MB


In [9]:
combined_df.isnull().sum()

Temperature      0
Humidity         0
Precipitation    0
WindSpeed        0
WindDirection    0
Date             0
Station          0
dtype: int64

In [10]:
combined_df.to_csv(root / "data" / "processed" / "weather_data.csv", index=False)

print("✅ Success: Weather data saved.")

✅ Success: Weather data saved.


In [10]:
import os
os.getcwd()


'd:\\Nueva carpeta\\AI-Based-Smart-City-Air-Quality-Monitoring-and-Forecasting-System-for-Greater-Bilbao\\notebooks'

In [12]:
weather_df = pd.read_csv("../data/processed/weather_data.csv")
air_quality_df = pd.read_csv("../data/processed/cleaned_air_quality_bilbao_2015_2026.csv")
print(weather_df.head())
print(air_quality_df.head())

   Temperature  Humidity  Precipitation  WindSpeed  WindDirection        Date  \
0          6.4        85            0.0        8.0            176  2015-01-01   
1          6.7        85            0.0        8.0            176  2015-01-01   
2          4.7        82            0.0        8.0            176  2015-01-01   
3          6.8        85            0.0        8.0            176  2015-01-01   
4          6.8        83            0.0        8.0            176  2015-01-01   

          Station  
0  ALGORTA_BBIZI2  
1       BARAKALDO  
2         BASAURI  
3         ERANDIO  
4       MAZARREDO  
         Date         station   Town Province   Latitude  Longitude   NO2  \
0  2015-01-01  ALGORTA_BBIZI2  Getxo  Bizkaia  43.362056  -3.022782  47.0   
1  2015-01-02  ALGORTA_BBIZI2  Getxo  Bizkaia  43.362056  -3.022782  56.0   
2  2015-01-03  ALGORTA_BBIZI2  Getxo  Bizkaia  43.362056  -3.022782  48.0   
3  2015-01-04  ALGORTA_BBIZI2  Getxo  Bizkaia  43.362056  -3.022782  43.0   
4  2015-

In [13]:
weather_df["Date"] = pd.to_datetime(weather_df["Date"])
air_quality_df["Date"] = pd.to_datetime(air_quality_df["Date"])

In [14]:
air_quality_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 29008 entries, 0 to 29007
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       29008 non-null  datetime64[us]
 1   station    29008 non-null  str           
 2   Town       29008 non-null  str           
 3   Province   29008 non-null  str           
 4   Latitude   29008 non-null  float64       
 5   Longitude  29008 non-null  float64       
 6   NO2        29008 non-null  float64       
 7   PM10       29008 non-null  float64       
 8   PM2.5      29008 non-null  float64       
 9   SO2        29008 non-null  float64       
dtypes: datetime64[us](1), float64(6), str(3)
memory usage: 2.8 MB


In [26]:
air_quality_df.isnull().sum()

Date         0
station      0
Town         0
Province     0
Latitude     0
Longitude    0
NO2          0
PM10         0
PM2.5        0
SO2          0
Year         0
Month        0
dtype: int64

In [15]:
weather_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 29008 entries, 0 to 29007
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Temperature    29008 non-null  float64       
 1   Humidity       29008 non-null  int64         
 2   Precipitation  29008 non-null  float64       
 3   WindSpeed      29008 non-null  float64       
 4   WindDirection  29008 non-null  int64         
 5   Date           29008 non-null  datetime64[us]
 6   Station        29008 non-null  str           
dtypes: datetime64[us](1), float64(3), int64(2), str(1)
memory usage: 1.8 MB


In [27]:
weather_df.isnull().sum()

Temperature      0
Humidity         0
Precipitation    0
WindSpeed        0
WindDirection    0
Date             0
Station          0
dtype: int64

In [17]:
# Number of rows
print(f"Number of rows of air_quality_df: {air_quality_df.shape[0]}")
print(f"Number of rows of weather_df: {weather_df.shape[0]}")

Number of rows of air_quality_df: 29008
Number of rows of weather_df: 29008


In [32]:
air_quality_df['station'] = air_quality_df['station'].replace('SANTURCE', 'SANTURTZI')

In [33]:
# ۱. ادغام دیتافریم‌ها
merged_df = pd.merge(
    air_quality_df, 
    weather_df, 
    left_on=["Date", "station"], 
    right_on=["Date", "Station"], 
    how="inner" 
)


# ۲. حذف ستون تکراری Station (چون station و Station هر دو در نتیجه هستند)
merged_df = merged_df.drop(columns=["Station"])

# ۳. تنظیم مجدد ساختار ستون‌ها برای خوانایی بهتر
# آوردن اطلاعات ایستگاه به ابتدای جدول
cols = ['Date', 'station', 'Town', 'Province', 'Latitude', 'Longitude'] + \
       ['NO2', 'PM10', 'PM2.5', 'SO2', 'Temperature', 'Humidity', 'Precipitation', 'WindSpeed', 'WindDirection']
merged_df = merged_df[cols]

# ۴. بررسی نهایی صحت ادغام
print(f"تعداد ردیف‌های نهایی: {merged_df.shape[0]}")


تعداد ردیف‌های نهایی: 29008


In [34]:
merged_df.isnull().sum()

Date             0
station          0
Town             0
Province         0
Latitude         0
Longitude        0
NO2              0
PM10             0
PM2.5            0
SO2              0
Temperature      0
Humidity         0
Precipitation    0
WindSpeed        0
WindDirection    0
dtype: int64

In [1]:
print(merged_df)

NameError: name 'merged_df' is not defined

In [38]:
merged_df.to_csv("../data/processed/air_quality_weather.csv", index=False)
print("✅ Success: Merged data saved.")

✅ Success: Merged data saved.


In [5]:
df = pd.read_csv("../data/processed/air_quality_weather.csv")
print(df.head())

         Date         station   Town Province   Latitude  Longitude   NO2  \
0  2015-01-01  ALGORTA_BBIZI2  Getxo  Bizkaia  43.362056  -3.022782  47.0   
1  2015-01-02  ALGORTA_BBIZI2  Getxo  Bizkaia  43.362056  -3.022782  56.0   
2  2015-01-03  ALGORTA_BBIZI2  Getxo  Bizkaia  43.362056  -3.022782  48.0   
3  2015-01-04  ALGORTA_BBIZI2  Getxo  Bizkaia  43.362056  -3.022782  43.0   
4  2015-01-05  ALGORTA_BBIZI2  Getxo  Bizkaia  43.362056  -3.022782  29.0   

   PM10  PM2.5  SO2  Temperature  Humidity  Precipitation  WindSpeed  \
0  25.0   28.0  9.0          6.4        85            0.0        8.0   
1  24.0   18.0  8.0          8.3        81            0.0        8.9   
2  33.0   21.0  8.0          8.9        80            0.0       13.0   
3  31.0   23.0  7.0          9.6        88            0.0        8.7   
4  18.0   11.0  5.0          9.0        85            0.0       13.9   

   WindDirection  
0            176  
1            207  
2            215  
3            219  
4        

In [39]:
merged_df.to_parquet("../data/processed/air_quality_weather.parquet", index=False)
print("✅ Success: Merged data saved in Parquet format.")

✅ Success: Merged data saved in Parquet format.
